In [ ]:
import torch
import json

from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left")

In [ ]:
def extract_ground_truth(answer_text):
    if "####" not in answer_text:
        return answer_text.strip() 
    return answer_text.split("####")[-1].strip()

In [ ]:
def extract_problem_concepts(model, tokenizer, data):
    results = []

    for i, example in enumerate(tqdm(data)):
        question_id = example["question_id"]
        question = example["question"]
        target_answer = extract_ground_truth(example["answer"])

        messages = [
            {
                "role": "system",
                "content": (
                    "You are a math problem analyzer. "
                    "You do NOT solve math problems or compute answers. "
                )
            },
            {
                "role": "user",
                "content": f"""
Analyze the following math problem.

DO NOT solve the problem.
DO NOT perform any calculations.
DO NOT output the final answer.

Identify the mathematical concepts required to solve the problem.

Examples of mathematical concepts include arithmetic operations, comparison,
ratios, fractions, percentages, equations, formulas, and unit conversion, etc...

Explain concisely your reasoning using only qualitative descriptions.

Text:
\"\"\"
{question}
\"\"\"
"""
            }
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer([text], return_tensors="pt").to(model.device)

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False,
                repetition_penalty=1.1,
            )

        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
        ]
    
        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        results.append({
            "question_id": question_id, 
            "question": question,
            "target_answer": target_answer,
            "concepts": response 
        })

    return results

In [ ]:
train_data = load_dataset("hyunjaehyun/gsm8k_train_subsets_shuffled")

In [ ]:
subset_1 = train_data["subset_1"]
subset_2 = train_data["subset_2"]
subset_3 = train_data["subset_3"]
subset_4 = train_data["subset_4"]

In [ ]:
print(subset_1)

Dataset({
    features: ['question', 'answer', 'question_id'],
    num_rows: 1868
})


In [ ]:
subsets = [subset_1, subset_2, subset_3, subset_4]

for i, subset in enumerate(subsets, start=1):
    subset_concepts = extract_problem_concepts(model, tokenizer, subset)

    with open(f"subset-{i}_math_concepts_results.json", "w", encoding="utf-8") as f:
        json.dump(subset_concepts, f, indent=2)

In [ ]:
def formulate_and_solve(model, tokenizer, data):
    results = []

    for i, example in enumerate(tqdm(data)): # data: List[Dict]
        question_id = example["question_id"]
        question = example["question"]
        concepts = example["concepts"]
        target_answer = example["target_answer"]

        messages = [
            {
                "role": "system",
                "content": (
                    "You are a math problem solver. "
                    "Explicitly define variables, formulate the necessary mathematical relations, and then compute the final answer. "
                    )
            },
            {
                "role": "user",
                "content": f"""
Solve the math problem using the procedure below.

Procedure:
1. Use the problem text as the primary source of information.
2. Use the given concepts only as guidance. The problem text always takes precedence.
3. Identify the quantities and define variables for the unknown quantities when needed.
4. Write the equations or symbolic relations needed to solve the problem.
5. Use these relations to compute the final answer.

Keep each step concise and do not include unnecessary explanations.

---

Problem:
\"\"\"
{question}
\"\"\"

Guiding concepts:
{concepts}
"""
            }
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer([text], return_tensors="pt").to(model.device)

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=1024,
                do_sample=False,
                repetition_penalty=1.1,
            )

        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
        ]
    
        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        results.append({
            "question_id": question_id, 
            "question": question,
            "concepts": concepts,
            "model_solving": response,
            "target_answer": target_answer
        })


    return results

In [ ]:
subsets_concepts = {}

for i in range(1, 5):
    with open(f"subset-{i}_math_concepts_results.json", "r", encoding="utf-8") as f:
        subsets_concepts[i] = json.load(f)

In [ ]:
"""
정상 작동 되는지. 원하는 대로 응답하는지 확인용
"""

import random

# question 길이 기준 내림차순 
sorted_results = sorted(
    subsets_concepts[1],
    key=lambda x: len(x.get("question", "")),
    reverse=True
)

# 상위 N개 중에서 랜덤 추출
N = 10  
sampled = random.sample(sorted_results[:N], 10)

In [ ]:
sampled_concepts = formulate_and_solve(model, tokenizer, sampled)

  0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
for i in range(1, 5):
    subset_solving = formulate_and_solve(model, tokenizer, subsets_concepts[i])

    with open(f"subset_{i}_student_solving.json", "w", encoding="utf-8") as f:
        json.dump(subset_solving, f, indent=2)